# autoresearch-web analysis

Analyze the 14-column `results.tsv` produced by the CRO inner + outer loops.

- **Inner loop** rows have `status` in `{proposed, pre_validated, pushed, discarded, crash}` and fill the pre-validation scores (`heuristic_score`, `judge_score`, `persona_score`, `lh_score`, `composite`).
- **Outer loop** rows have `status` in `{measuring, winner, loser}` and fill `measured_lift`, `measured_ci_low`, `measured_ci_high`.
- The same `variant_slug` can appear multiple times (once per status change). We group by slug and take the latest row per slug for 'current state' views, but keep the full history for progress plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

COLS = [
    'variant_slug', 'hypothesis_source', 'diff_lines', 'status',
    'lh_score', 'judge_score', 'persona_score', 'heuristic_score',
    'composite', 'experiment_id',
    'measured_lift', 'measured_ci_low', 'measured_ci_high',
    'description',
]

df = pd.read_csv('results.tsv', sep='\t', dtype=str, keep_default_na=False)
for c in ['diff_lines']:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
for c in ['lh_score', 'judge_score', 'persona_score', 'heuristic_score',
          'composite', 'measured_lift', 'measured_ci_low', 'measured_ci_high']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['status'] = df['status'].str.strip().str.lower()
df['row_idx'] = np.arange(len(df))

print(f'Total rows: {len(df)}')
print(f'Unique variants: {df["variant_slug"].nunique()}')
df.head(10)

## Status breakdown

In [ ]:
current = df.sort_values('row_idx').drop_duplicates('variant_slug', keep='last')
counts = current['status'].value_counts()
print('Current status (latest row per slug):')
print(counts.to_string())

pushed_or_later = current['status'].isin(['pushed', 'measuring', 'winner', 'loser']).sum()
pre_val_only = (current['status'] == 'pre_validated').sum()
print(f'\nPushed or beyond: {pushed_or_later}')
print(f'Pre-validated (held): {pre_val_only}')
print(f'Winners: {(current["status"] == "winner").sum()}')

## Inner-loop progress: composite score over iteration

Each point is one inner-loop iteration. Running max of `composite` is the pre-validation frontier — the best *predicted* variant so far. Remember this is a prediction, not a measured lift.

In [ ]:
inner_statuses = {'proposed', 'pre_validated', 'pushed', 'discarded', 'crash'}
inner = df[df['status'].isin(inner_statuses)].copy()
inner = inner.dropna(subset=['composite']).reset_index(drop=True)

if len(inner) > 0:
    fig, ax = plt.subplots(figsize=(14, 7))

    color_map = {
        'discarded':     '#cccccc',
        'pre_validated': '#f1c40f',
        'pushed':        '#2ecc71',
        'crash':         '#e74c3c',
        'proposed':      '#3498db',
    }
    for status, color in color_map.items():
        sub = inner[inner['status'] == status]
        if len(sub) == 0:
            continue
        ax.scatter(sub.index, sub['composite'], c=color, s=45,
                   alpha=0.8, label=status, edgecolors='black', linewidths=0.3)

    running_max = inner['composite'].cummax()
    ax.step(inner.index, running_max, where='post', color='#16a085',
            linewidth=2, alpha=0.8, label='running max composite')

    ax.axhline(0, color='#888', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_xlabel('Inner-loop iteration')
    ax.set_ylabel('Composite score (higher is better)')
    ax.set_title('Inner loop: predicted variant quality over time')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print('No inner-loop rows with composite scores yet.')

## Calibration: composite vs measured lift

This is the single most important plot in the framework. It tells you whether your LLM-judge + heuristic composite score correlates with real measured A/B test lift. If the scatter is all over the place, your pre-validation is miscalibrated and `skills/rank.md` should be re-weighting signals based on recent history.

Only plots variants that have both a composite (inner loop) AND a measured_lift (outer loop).

In [ ]:
by_slug = df.sort_values('row_idx').groupby('variant_slug')

rows = []
for slug, g in by_slug:
    composite = g['composite'].dropna()
    lift = g['measured_lift'].dropna()
    if len(composite) > 0 and len(lift) > 0:
        rows.append({
            'slug': slug,
            'composite': composite.iloc[0],
            'measured_lift': lift.iloc[0],
            'ci_low': g['measured_ci_low'].dropna().iloc[0] if g['measured_ci_low'].notna().any() else np.nan,
            'ci_high': g['measured_ci_high'].dropna().iloc[0] if g['measured_ci_high'].notna().any() else np.nan,
            'status': g['status'].iloc[-1],
        })

calib = pd.DataFrame(rows)

if len(calib) >= 2:
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = calib['status'].map({'winner': '#2ecc71', 'loser': '#e74c3c',
                                    'measuring': '#f39c12'}).fillna('#7f8c8d')
    yerr = np.array([
        calib['measured_lift'] - calib['ci_low'],
        calib['ci_high'] - calib['measured_lift'],
    ])
    ax.errorbar(calib['composite'], calib['measured_lift'], yerr=yerr,
                fmt='o', ecolor='#bbb', capsize=3, markersize=0, alpha=0.6)
    ax.scatter(calib['composite'], calib['measured_lift'], c=colors, s=90,
               edgecolors='black', linewidths=0.5, zorder=3)
    ax.axhline(0, color='#888', linestyle='--', alpha=0.5)
    ax.axvline(0, color='#888', linestyle='--', alpha=0.5)
    if len(calib) >= 3:
        z = np.polyfit(calib['composite'], calib['measured_lift'], 1)
        xs = np.linspace(calib['composite'].min(), calib['composite'].max(), 50)
        ax.plot(xs, np.polyval(z, xs), color='#16a085', linewidth=1.5,
                alpha=0.6, label=f'linear fit (slope={z[0]:.3f})')
        corr = calib['composite'].corr(calib['measured_lift'])
        ax.set_title(f'Calibration: composite vs measured lift (r = {corr:.2f}, n = {len(calib)})')
    else:
        ax.set_title(f'Calibration: composite vs measured lift (n = {len(calib)})')
    ax.set_xlabel('Pre-validation composite (predicted)')
    ax.set_ylabel('Measured lift (real A/B test)')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print(f'Not enough variants with both composite and measured_lift yet (need >= 2, have {len(calib)}).')
    print('Run real experiments via the outer loop to populate this plot.')

## Top candidates (ranked)

Current ranking uses `measured_lift` where available, else `composite`. Mirrors what `skills/rank.md` writes to `variants/RANKED.md`.

In [ ]:
current = df.sort_values('row_idx').drop_duplicates('variant_slug', keep='last').copy()
current['rank_score'] = current['measured_lift'].fillna(current['composite'])
current = current.sort_values('rank_score', ascending=False).reset_index(drop=True)

alive = current[current['status'].isin(
    ['pre_validated', 'pushed', 'measuring', 'winner']
)]

print(f"Top {min(20, len(alive))} candidates:")
print(f"{'rank':>4}  {'slug':<40}  {'status':<14}  {'score':>8}  {'diff':>5}  description")
print('-' * 120)
for i, row in alive.head(20).iterrows():
    score = row['rank_score']
    score_str = f'{score:+.3f}' if pd.notna(score) else '   --  '
    diff = row['diff_lines']
    diff_str = f'{int(diff):>5}' if pd.notna(diff) else '   --'
    desc = str(row['description'])[:60]
    print(f'{i+1:>4}  {row["variant_slug"]:<40}  {row["status"]:<14}  {score_str:>8}  {diff_str}  {desc}')

## Hypothesis source breakdown

Which adapters / strategies are producing the hypotheses? Mode collapse looks like one tag dominating.

In [ ]:
sources = []
for src in df['hypothesis_source'].dropna():
    for tag in str(src).split(','):
        tag = tag.strip()
        if tag:
            sources.append(tag)
src_counts = pd.Series(sources).value_counts().head(20)
print('Top hypothesis sources:')
print(src_counts.to_string())

if len(src_counts) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, len(src_counts) * 0.3)))
    src_counts.plot.barh(ax=ax, color='#3498db')
    ax.set_xlabel('Variants citing this source')
    ax.set_title('Hypothesis source distribution')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()